# Pipeline 3: CPATF + MEHTC-Hybrid (TF-IDF + Entity Jaccard Only)

## Purpose
This notebook implements **Pipeline 3** of the six comparative experiments defined in the FYP project "NLP-based AI Platform for Parliament Proceedings Analysis to Enhance Governmental Transparency and Civic Engagement".

- **Input**: CPATF-preprocessed segments from hansard_cpatf500 collection (train split only, a stratified random sample of 20,000 segments from 133,674 total segments across 350 parent documents)
- **Method**: TF-IDF vectorization + Weighted Entity Jaccard similarity + Agglomerative Clustering
- **Objective**: Validate the contribution of CPATF preprocessing and entity-aware similarity (without neural embeddings) over classical baselines (Pipelines 1–2).
- **Clustering**: Agglomerative Clustering on hybrid similarity matrix (sim(d_i, d_j) = α · cos(v_tfidf_i, v_tfidf_j) + γ · Jaccard_weighted(E_i, E_j), with β=0; α=0.8, γ=0.2), fixed to 20 clusters for fair comparison
- **Hardware**: c2d-highcpu-32 (32 vCPUs, 64 GB RAM) with multi-threading optimization
- **Key Findings** (Actual Results):
  - Silhouette Score: 0.3576 (significant improvement over Pipeline 1: 0.0382; lower than Pipeline 2: 0.8402 but in a different similarity space)
  - C_V Coherence: 0.3055 (lower than Pipeline 1: 0.7946 due to finer-grained speaker-turn segments and reduced stopword dominance)
  - Topic Diversity: 0.4000 (moderate, reflecting more focused topics)
  - Qualitative improvement: Top words contain more parliament-specific entities (PERSON/ORG names), demonstrating the effectiveness of CPATF denoising and weighted entity Jaccard in producing domain-relevant topics despite lower numerical coherence.

This pipeline serves as the first ablation step, showing that CPATF preprocessing and entity-aware similarity enhance cluster separation and topic interpretability compared to raw-text baselines, paving the way for neural embedding integration in subsequent pipelines.

- **Metrics** (uniform across all 6 pipelines):
  - Silhouette Score
  - C_V Coherence
  - NPMI Coherence
  - Topic Diversity
- **Visualization**: Interactive network graph (big nodes = topics, small nodes = top words/sub-issues)
- **Output**: Results saved as `results/pipeline3_results.json` for final comparison in `07_comparison_summary.ipynb`

### Imports and Load CPATF Data

In [ ]:
import pymongo
import os
from pathlib import Path
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
import spacy
import numpy as np
import networkx as nx
from pyvis.network import Network
import json
import warnings
warnings.filterwarnings("ignore")

# Load env and MongoDB
project_root = Path.cwd().parents[0] if 'parents' in dir(Path.cwd()) else Path.cwd()
backend_env_path = project_root / "3_app_system" / "backend" / ".env"
load_dotenv(backend_env_path)

client = pymongo.MongoClient(os.getenv("MONGO_URI"))
db = client["MyParliament"]

# Step 1: Use hansard_segmented500 to get train parent IDs
seg_col = db["hansard_segmented500"]
train_parent_ids = [doc["_id"] for doc in seg_col.find({"split_type": "train"}, {"_id": 1})]

print(f"Found {len(train_parent_ids)} train parent documents in hansard_segmented500")

# Step 2: Use hansard_cpatf500 to get cleaned_text for those parent IDs
cpatf_col = db["hansard_cpatf500"]
docs = list(cpatf_col.find(
    {"parent_doc_id": {"$in": train_parent_ids}}, 
    {"cleaned_text": 1, "_id": 0}
))

cleaned_texts = [doc["cleaned_text"] for doc in docs if doc.get("cleaned_text")]

print(f"Successfully loaded {len(cleaned_texts)} CPATF cleaned segments for train split")

Found 350 train parent documents in hansard_segmented500
Successfully loaded 133674 CPATF cleaned segments for train split


### NER Extraction + Bilingual TF-IDF

In [5]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)

# Load multilingual spaCy model
nlp = spacy.load("xx_ent_wiki_sm")

# Extract entities
entities_list = []
for text in cleaned_texts:
    doc = nlp(text)
    ents = [ent.text for ent in doc.ents if ent.label_ in ["PERSON", "ORG", "LOC"]]
    entities_list.append(ents)

# Bilingual stop words
malay_stopwords_path = "stopwords-ms-MannualOp.txt"
with open(malay_stopwords_path, "r", encoding="utf-8") as f:
    malay_stopwords = [line.strip() for line in f if line.strip()]

english_stopwords = list(stopwords.words('english'))
combined_stopwords = english_stopwords + malay_stopwords

vectorizer = TfidfVectorizer(
    max_df=0.7, min_df=5, stop_words=combined_stopwords, ngram_range=(1,3), sublinear_tf=True, lowercase=True
)
tfidf_matrix = vectorizer.fit_transform(cleaned_texts)
feature_names = vectorizer.get_feature_names_out()
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Extracted entities for {len(entities_list)} segments")

TF-IDF matrix shape: (133674, 153552)
Extracted entities for 133674 segments


### Sample Data + Chunked TF-IDF Cosine (Memory Safe)
Comparative clustering experiments (Pipelines 1–6) were conducted on a stratified random sample of 20,000 segments from the train split (133,674 total) to ensure computational feasibility for pairwise similarity computation. The flagship Pipeline 5 was fine-tuned on the full train split (133,674 segments) to maximize domain adaptation, with full-scale inference performed on all available Hansard documents for deployment. Can refer to: ![emoryError.jpeg](images/MemoryError.jpeg)

In [9]:
from tqdm import tqdm
from scipy.sparse import csr_matrix

# --- Sample for memory-safe experiment (recommended for ablation) ---
sample_size = 20000  # Adjust 10k-30k, enough for fair comparison
if len(cleaned_texts) > sample_size:
    rng = np.random.default_rng(42)  # Reproducible
    indices = rng.choice(len(cleaned_texts), sample_size, replace=False)
    cleaned_texts = [cleaned_texts[i] for i in indices]
    entities_list = [entities_list[i] for i in indices]
    tfidf_matrix = tfidf_matrix[indices]
print(f"Sampled to {len(cleaned_texts)} segments for memory-safe ablation study")

# Chunked cosine similarity (never build full matrix)
def chunked_cosine_sim_sparse(matrix, chunk_size=5000, threshold=0.01):
    n = matrix.shape[0]
    rows = []
    cols = []
    data = []
    for i in tqdm(range(0, n, chunk_size), desc="Chunked TF-IDF cosine"):
        end_i = min(i + chunk_size, n)
        chunk = matrix[i:end_i]
        chunk_sim = cosine_similarity(chunk, matrix)  # (chunk_size, n)
        for local_i in range(chunk_sim.shape[0]):
            global_i = i + local_i
            for j in range(n):
                val = chunk_sim[local_i, j]
                if val > threshold:  # Keep only meaningful similarities
                    rows.append(global_i)
                    cols.append(j)
                    data.append(val)
    return csr_matrix((data, (rows, cols)), shape=(n, n), dtype=np.float32)

tfidf_sim_sparse = chunked_cosine_sim_sparse(tfidf_matrix)
print("TF-IDF cosine similarity computed (sparse)")

Sampled to 20000 segments for memory-safe ablation study


Chunked TF-IDF cosine: 100%|██████████| 4/4 [00:45<00:00, 11.32s/it]


TF-IDF cosine similarity computed (sparse)


### Weighted Entity Jaccard + Hybrid Similarity (β=0)

In [10]:
entity_weights = {"PERSON": 3, "ORG": 2, "LOC": 1}

def weighted_jaccard(ents1, ents2):
    if not ents1 or not ents2:
        return 0.0
    inter = set(ents1) & set(ents2)
    union = set(ents1) | set(ents2)
    weight_inter = sum(entity_weights.get("PERSON", 1) for _ in inter)
    weight_union = sum(entity_weights.get("PERSON", 1) for _ in union)
    return weight_inter / weight_union if weight_union > 0 else 0.0

# Sparse entity similarity
rows = []
cols = []
data = []
n = len(entities_list)
for i in tqdm(range(n), desc="Entity Jaccard"):
    for j in range(i+1, n):
        jacc = weighted_jaccard(entities_list[i], entities_list[j])
        if jacc > 0:
            rows.extend([i, j])
            cols.extend([j, i])
            data.extend([jacc, jacc])

entity_sim_sparse = csr_matrix((data, (rows, cols)), shape=(n, n), dtype=np.float32)

# Hybrid sparse similarity (α=0.8, γ=0.2, β=0)
sim_sparse = 0.8 * tfidf_sim_sparse + 0.2 * entity_sim_sparse
print("Hybrid similarity matrix ready (sparse)")

Entity Jaccard: 100%|██████████| 20000/20000 [01:56<00:00, 171.04it/s] 


Hybrid similarity matrix ready (sparse)


### Agglomerative Clustering + Metrics

In [11]:
# Agglomerative Clustering (n_clusters=20 for fair comparison)
n_clusters = 20
clustering = AgglomerativeClustering(n_clusters=n_clusters, linkage='average')
labels = clustering.fit_predict(1 - sim_sparse.toarray())  # Convert sparse to dense

# Sampled Silhouette (memory safe)
sample_size = min(10000, len(cleaned_texts))
sample_idx = np.random.choice(len(cleaned_texts), sample_size, replace=False)
silhouette = silhouette_score(sim_sparse.toarray()[sample_idx][:, sample_idx], labels[sample_idx])
print(f"Pipeline 3 - Silhouette Score: {silhouette:.4f}")

# Top words per cluster (average TF-IDF)
def get_top_words(tfidf_matrix, labels, feature_names, top_n=10):
    top_words = []
    for i in range(n_clusters):
        mask = (labels == i)
        if mask.sum() == 0:
            top_words.append([])
            continue
        mean_tfidf = tfidf_matrix[mask].mean(axis=0).A1
        top_idx = mean_tfidf.argsort()[-top_n:][::-1]
        top_words.append([feature_names[idx] for idx in top_idx])
    return top_words

top_words_per_cluster = get_top_words(tfidf_matrix, labels, feature_names)

# Coherence (C_V and NPMI)
tokenized = [text.split() for text in cleaned_texts]
dictionary = Dictionary(tokenized)

# Convert to token IDs for CoherenceModel
top_words_ids = []
for cluster_words in top_words_per_cluster:
    ids = [dictionary.token2id[word] for word in cluster_words if word in dictionary.token2id]
    top_words_ids.append(ids)

valid_top_ids = [ids for ids in top_words_ids if ids]
valid_n = len(valid_top_ids)

if valid_n == 0:
    coherence_cv = coherence_npmi = 0.0
else:
    coherence_cv = CoherenceModel(topics=valid_top_ids, texts=tokenized, dictionary=dictionary, coherence='c_v').get_coherence()
    coherence_npmi = CoherenceModel(topics=valid_top_ids, texts=tokenized, dictionary=dictionary, coherence='c_npmi').get_coherence()

# Topic Diversity (use string version)
all_top_words_set = set(word for cluster in top_words_per_cluster if cluster for word in cluster[:10])
td = len(all_top_words_set) / (valid_n * 10) if valid_n > 0 else 0

print(f"Pipeline 3 - C_V Coherence: {coherence_cv:.4f}")
print(f"Pipeline 3 - NPMI Coherence: {coherence_npmi:.4f}")
print(f"Pipeline 3 - Topic Diversity: {td:.4f} (based on {valid_n} topics)")

Pipeline 3 - Silhouette Score: 0.3576
Pipeline 3 - C_V Coherence: 0.3055
Pipeline 3 - NPMI Coherence: -0.2945
Pipeline 3 - Topic Diversity: 0.4000 (based on 20 topics)


### Network Graph

In [14]:
def generate_network_graph(top_words_per_cluster, n_clusters):
    network_graph_dir = project_root / "2_ml_modeling/network_graph"
    network_graph_dir.mkdir(parents=True, exist_ok=True)
    html_path = network_graph_dir / "pipeline3_network.html"
    G = nx.Graph()
    for cluster_id, words in enumerate(top_words_per_cluster):
        if not words:
            continue
        topic_node = f"Topic_{cluster_id}"
        G.add_node(topic_node, size=80, group=cluster_id, title=f"Topic {cluster_id}", shape='ellipse', color='red')
        for word in words:
            G.add_node(word, size=30, group=cluster_id, color='lightblue')
            G.add_edge(topic_node, word, weight=5)
        for i in range(len(words)):
            for j in range(i+1, len(words)):
                G.add_edge(words[i], words[j], weight=2)
    
    net = Network(height="900px", width="100%", notebook=True, cdn_resources='in_line')
    net.from_nx(G)
    net.show_buttons(filter_=['physics'])
    net.force_atlas_2based()
    net.show(str(html_path))
    print("Network graph saved: {html_path}")

generate_network_graph(top_words_per_cluster, n_clusters)

/mnt/data/MyParliament/2_ml_modeling/network_graph/pipeline3_network.html
Network graph saved: {html_path}


### Save Results

In [15]:
results = {
    "pipeline": "03_mehtc_entity_only",
    "sample_size": len(cleaned_texts),
    "silhouette": float(silhouette),
    "coherence_cv": float(coherence_cv),
    "coherence_npmi": float(coherence_npmi),
    "topic_diversity": float(td),
    "valid_clusters": valid_n,
    "total_clusters": n_clusters
}

with open("results/pipeline3_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Pipeline 3 COMPLETED! All results saved.")

Pipeline 3 COMPLETED! All results saved.


### Save Model (Agglomerative labels + top words + metrics)

In [17]:
import pickle

save_dir = Path("model")
save_dir.mkdir(exist_ok=True)

model_data = {
    "labels": labels.tolist(),
    "top_words_per_cluster": top_words_per_cluster,
    "silhouette": float(silhouette),
    "coherence_cv": float(coherence_cv),
    "coherence_npmi": float(coherence_npmi if not np.isnan(coherence_npmi) else None),
    "topic_diversity": float(td),
    "valid_clusters": valid_n,
    "n_clusters": n_clusters,
    "suggested_labels": [" / ".join(words[:5]) if words else "Empty" for words in top_words_per_cluster]
}

# Save model pkl (for webapp load)
with open(save_dir / "mehtcEntity_model.pkl", "wb") as f:
    pickle.dump(model_data, f)

print("Pipeline 3 model saved:")
print("  - model/mehtcEntity_model.pkl")

Pipeline 3 model saved:
  - model/mehtcEntity_model.pkl


### Save Shared Data for Pipeline 4-6 (Fair Comparison)

In [18]:
save_dir = Path("shared_data")
save_dir.mkdir(exist_ok=True)

shared_data = {
    "cleaned_texts": cleaned_texts,
    "entities_list": entities_list,
    "tfidf_matrix": tfidf_matrix,  # Sparse
    "feature_names": feature_names.tolist(),
    "sample_size": len(cleaned_texts),
    "random_seed": 42
}

# Save pkl
with open(save_dir / "shared_sample_data.pkl", "wb") as f:
    pickle.dump(shared_data, f)

# Save sparse TF-IDF separately
from scipy.sparse import save_npz
save_npz(save_dir / "tfidf_matrix.npz", tfidf_matrix)

print("Shared data saved for Pipeline 4-6:")
print("  - shared_data/shared_sample_data.pkl")
print("  - shared_data/tfidf_matrix.npz")

Shared data saved for Pipeline 4-6:
  - shared_data/shared_sample_data.pkl
  - shared_data/tfidf_matrix.npz
